# 01_descarga_nisar

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
from pathlib import Path

SEP = "=" * 40

# ---- Folders ----
DIR_BASE = next((c for c in (Path.cwd(), Path.cwd().parent) if (c / "metadatos").exists()), Path(".."))
DIR_META = DIR_BASE / "metadatos"
DIR_H5   = DIR_BASE / "nisar_gcov_h5"
DIR_MALOS = DIR_H5 / "_incompletos"
for d in (DIR_BASE, DIR_H5):
    d.mkdir(parents=True, exist_ok=True)

LON_MIN, LON_MAX = -77.05, -76.45
LAT_MIN, LAT_MAX = 5.55, 7.20
AOI_WKT = (f"POLYGON(({LON_MIN} {LAT_MIN},{LON_MAX} {LAT_MIN},{LON_MAX} {LAT_MAX},"
           f"{LON_MIN} {LAT_MAX},{LON_MIN} {LAT_MIN}))")

FECHA_INI = datetime(2025, 11, 1)
FECHA_FIN = datetime(2026, 10, 15)

VENTANAS = [
    (datetime(2026, 1, 19), datetime(2026, 1, 31)),
    (datetime(2026, 2, 1),  datetime(2026, 2, 12)),
    (datetime(2026, 5, 4),  datetime(2026, 5, 15)),
    (datetime(2026, 5, 29), datetime(2026, 6, 10)),
    (datetime(2026, 7, 19), datetime(2026, 7, 31)),
    (datetime(2026, 8, 28), datetime(2026, 9, 8)),
]
EVENTOS = ["Murindó", "Vigía del Fuerte", "Río Quito / Quibdó",
           "Quibdó, Río Quito, Cértegui", "Medio Atrato (Beté)", "Quibdó (post-sismo)"]
MARGEN_PREVIO = timedelta(days=5)
POST_MAX_DIAS = 30

ORBITAS_A_BAJAR = ["Ascendente", "Descendente"]
STACKS_A_BAJAR  = None
MAX_DESCARGA_GB = 120
PREFERIR_DOBLE_POL = True
BORRAR_INCOMPLETOS = False

print(SEP)
print(f"{'Zona':<12}: lon [{LON_MIN}, {LON_MAX}]  lat [{LAT_MIN}, {LAT_MAX}]")
print(f"{'Periodo':<12}: {FECHA_INI:%Y-%m-%d} a {FECHA_FIN:%Y-%m-%d}")
print(f"{'Órbitas':<12}: {ORBITAS_A_BAJAR}")
print(f"{'Salida':<12}: {DIR_H5}")
print(SEP)

Zona        : lon [-77.05, -76.45]  lat [5.55, 7.2]
Periodo     : 2025-11-01 a 2026-10-15
Órbitas     : ['Ascendente', 'Descendente']
Salida      : C:\Users\Usuario\Desktop\NASA SAC\nisar_gcov_h5


## 2. Earthdata login

In [3]:
import os
import getpass
import asf_search as asf

URL_AUTORIZAR = "https://urs.earthdata.nasa.gov/approve_app?client_id=BO_n7nTIlMljdvU6kRRB3g"

def iniciar_sesion():
    token = os.environ.get("EARTHDATA_TOKEN", "").strip()
    if token:
        print("Usando EARTHDATA_TOKEN de las variables de entorno.")
        return asf.ASFSession().auth_with_token(token)

    modo = input("¿Entrar con [1] usuario/contraseña o [2] token? ").strip()
    if modo == "2":
        tk = getpass.getpass("Token Earthdata: ").strip()
        return asf.ASFSession().auth_with_token(tk)
    usuario = input("Usuario Earthdata (no el correo): ").strip()
    clave = getpass.getpass("Contraseña Earthdata: ")
    return asf.ASFSession().auth_with_creds(usuario, clave)

try:
    session = iniciar_sesion()
    print("Sesión iniciada.")
except Exception as e:
    print(SEP)
    print(f"No se pudo iniciar sesión: {type(e).__name__}")
    if "Pre authorization required" in str(e) or "approve_app" in str(e):
        print("Falta autorizar la app de ASF en su cuenta. Abran este enlace, acepten y vuelvan a correr la celda:")
        print(f"  {URL_AUTORIZAR}")
    else:
        print(e)
        print("Revisen el usuario (no el correo) y la contraseña, o usen la opción [2] token.")
    print(SEP)
    raise

Sesión iniciada.


## 3. Search scenes and identify orbit, track, frame, type and mode

In [4]:
import re
import json
import pandas as pd

PATRON = re.compile(r"NISAR_L2_([A-Z]{2})_GCOV_(\d{3})_(\d{3})_([AD])_(\d{3})_(\d{4})_([A-Z]{4})_[A-Z]_(\d{8}T\d{6})")
NOMBRE_ORBITA = {"A": "Ascendente", "D": "Descendente"}

def parsear_nombre(nombre):
    m = PATRON.search(nombre or "")
    if not m:
        return {}
    tipo, ciclo, track, direc, frame, modo, pols, t = m.groups()
    fecha = pd.to_datetime(t, format="%Y%m%dT%H%M%S")
    return {"fecha": fecha, "dia": fecha.normalize(),
            "hora_local": (fecha - pd.Timedelta(hours=5)).strftime("%H:%M"),
            "orbita": NOMBRE_ORBITA[direc], "dir": direc, "track": int(track), "frame": int(frame),
            "tipo": tipo, "modo": modo, "pols": pols}


def tam_gb(props, archivo):
    b = props.get("bytes")
    if isinstance(b, dict):
        v = b.get(archivo)
        if v is None:
            v = next((x for k, x in b.items() if k.endswith(".h5") and "QA_STATS" not in k), None)
        b = v.get("bytes") if isinstance(v, dict) else v
    try:
        return float(b) / 1e9
    except (TypeError, ValueError):
        return float("nan")


def en_ventana(fecha):
    return any(ini <= fecha <= fin + timedelta(days=1) for ini, fin in VENTANAS)


def cerca_de_evento(fecha):
    return any(ini - MARGEN_PREVIO <= fecha < ini for ini, fin in VENTANAS)


def nombre_archivo(url):
    return url.split("/")[-1]

In [ ]:
opts = asf.ASFSearchOptions(
    dataset=["NISAR"],
    processingLevel=["GCOV"],
    intersectsWith=AOI_WKT,
    start=FECHA_INI,
    end=FECHA_FIN,
    maxResults=1000,
    session=session,
)
res = asf.search(opts=opts)
urls_h5 = res.find_urls(extension=".h5", pattern=r"^(?!.*QA_STATS).*", directAccess=False)

filas = []
for p in res:
    pr = p.properties
    nombre = pr.get("sceneName") or pr.get("fileID") or ""
    info = parsear_nombre(nombre)
    if not info:
        continue
    url = next((u for u in urls_h5 if nombre in u), pr.get("url") or "")
    archivo = nombre_archivo(url)
    coleccion = re.search(r"(NISAR_L2_GCOV_[A-Z0-9_]+)/", url)
    filas.append({"escena": nombre, "url": url, "archivo": archivo, "gb": tam_gb(pr, archivo),
                  "coleccion": coleccion.group(1) if coleccion else "", **info})

todas = pd.DataFrame(filas)
if todas.empty:
    raise SystemExit("No hay escenas GCOV en la zona y el periodo. Revisen la cobertura en Vertex.")
todas["stack"] = todas["dir"] + "_T" + todas["track"].astype(str).str.zfill(3)

modo_dom = todas.groupby("stack")["modo"].agg(lambda s: s.value_counts().index[0])
todas["es_UR"]      = (todas["tipo"] == "UR").astype(int)
todas["es_simple"]  = (todas["pols"] == "SHSH").astype(int)
todas["otro_modo"]  = (todas["modo"] != todas["stack"].map(modo_dom)).astype(int)
todas = todas.sort_values(["stack", "dia", "frame", "es_UR", "es_simple", "otro_modo", "escena"],
                          ascending=[True, True, True, True, True, True, False])
inv = todas.drop_duplicates(["stack", "dia", "frame"], keep="first").copy()
descartadas = len(todas) - len(inv)

inv["en_ventana"] = inv["dia"].apply(en_ventana)
inv["justo_antes_de_evento"] = inv["dia"].apply(cerca_de_evento)
inv = inv.drop(columns=["es_UR", "es_simple", "otro_modo"]).sort_values(["stack", "fecha"]).reset_index(drop=True)

inv.to_csv(DIR_META / "inventario_nisar_choco.csv", index=False)
with open(DIR_META / "huellas_nisar_choco.geojson", "w") as f:
    json.dump(res.geojson(), f)

print(SEP)
print(f"{'Productos encontrados':<26}: {len(res)}")
print(f"{'Duplicados descartados':<26}: {descartadas}  (UR/PR, modos o versiones repetidas)")
print(f"{'Archivos únicos':<26}: {len(inv)}")
print(f"{'Tamaño total en ASF':<26}: {inv['gb'].sum():.1f} GB")
print(SEP)
display(inv[["fecha", "hora_local", "stack", "frame", "tipo", "modo", "pols", "coleccion", "en_ventana", "gb"]])

Productos encontrados     : 109
Duplicados descartados    : 29  (UR/PR, modos o versiones repetidas)
Archivos únicos           : 80
Tamaño total en ASF       : 389.0 GB


,fecha,hora_local,stack,frame,tipo,modo,pols,coleccion,en_ventana,gb
0,2026-06-17 10:38:36,05:38,A_T047,4,PR,4005,DHDH,NISAR_L2_GCOV_PROVISIONAL_V1,False,6.488588
1,2026-06-17 10:39:06,05:39,A_T047,5,PR,4005,DHDH,NISAR_L2_GCOV_PROVISIONAL_V1,False,7.289700
2,2026-06-29 10:38:31,05:38,A_T047,4,PR,4005,DHDH,NISAR_L2_GCOV_PROVISIONAL_V1,False,7.482638
3,2026-06-29 10:39:06,05:39,A_T047,5,PR,4005,DHDH,NISAR_L2_GCOV_PROVISIONAL_V1,False,7.092568
4,2026-07-11 10:38:31,05:38,A_T047,4,PR,4005,DHDH,NISAR_L2_GCOV_PROVISIONAL_V1,False,7.285506
...,...,...,...,...,...,...,...,...,...,...
75,2026-09-13 23:37:30,18:37,D_T112,86,PR,2005,SHSH,NISAR_L2_GCOV_PROVISIONAL_V1,False,0.956301
76,2026-09-25 23:36:56,18:36,D_T112,85,PR,2005,DHDH,NISAR_L2_GCOV_PROVISIONAL_V1,False,1.652556
77,2026-09-25 23:37:30,18:37,D_T112,86,PR,2005,DHDH,NISAR_L2_GCOV_PROVISIONAL_V1,False,1.337983
78,2026-10-07 23:36:56,18:36,D_T112,85,PR,4005,DHDH,NISAR_L2_GCOV_PROVISIONAL_V1,False,3.242197


## 4. Stacks and event coverage

In [6]:
pasadas = (inv.groupby(["stack", "orbita", "dia"])
             .agg(frames=("frame", lambda s: "+".join(f"F{x:03d}" for x in sorted(s))),
                  archivos=("archivo", list), gb=("gb", "sum"),
                  pols=("pols", lambda s: "/".join(sorted(set(s)))),
                  modos=("modo", lambda s: "/".join(sorted(set(s)))),
                  hora_local=("hora_local", "first"),
                  en_ventana=("en_ventana", "first"), justo_antes=("justo_antes_de_evento", "first"))
             .reset_index())

cab = "".join(f"{f'E{i}':>5}" for i in range(1, len(VENTANAS) + 1))
print(SEP)
print(f"{'Stack':<9}{'Órbita':<13}{'Hora':<7}{'Pasadas':>8}{cab}{'Libres':>8}  Desde → hasta")
for (stack, orbita), p in pasadas.groupby(["stack", "orbita"]):
    cuentas = [((p["dia"] >= ini) & (p["dia"] <= fin + timedelta(days=1))).sum() for ini, fin in VENTANAS]
    libres_n = (~p["en_ventana"] & ~p["justo_antes"]).sum()
    print(f"{stack:<9}{orbita:<13}{p['hora_local'].iloc[0]:<7}{len(p):>8}" + "".join(f"{c:>5}" for c in cuentas)
          + f"{libres_n:>8}  {p['dia'].min():%Y-%m-%d} → {p['dia'].max():%Y-%m-%d}")
print(SEP)
mezcla = pasadas[pasadas["modos"].str.contains("/") | pasadas["pols"].str.contains("SHSH")]
if len(mezcla):
    print("Ojo con estas pasadas (modos mezclados entre frames o solo HH):")
    for r in mezcla.itertuples():
        print(f"  {r.stack}  {r.dia:%Y-%m-%d}  {r.frames:<10} modos {r.modos:<10} pols {r.pols}")

Stack    Órbita       Hora    Pasadas   E1   E2   E3   E4   E5   E6  Libres  Desde → hasta
A_T047   Ascendente   05:38         9    0    0    0    0    1    2       6  2026-06-17 → 2026-10-03
A_T148   Ascendente   05:46         9    0    0    0    0    0    1       6  2026-06-24 → 2026-10-10
D_T011   Descendente  18:28        14    0    0    0    0    1    1      11  2025-11-22 → 2026-10-12
D_T112   Descendente  18:37         8    0    0    0    0    0    1       6  2026-06-21 → 2026-10-07
Ojo con estas pasadas (modos mezclados entre frames o solo HH):
  A_T148  2026-08-23  F004+F005  modos 2005       pols SHSH
  D_T112  2026-07-03  F085+F086  modos 2005/4005  pols DHDH/SHSH
  D_T112  2026-08-20  F085+F086  modos 2005/4005  pols DHDH/SHSH
  D_T112  2026-09-13  F085+F086  modos 2005/4005  pols DHDH/SHSH
  D_T112  2026-10-07  F085+F086  modos 2005/4005  pols DHDH/SHSH


## 5. Pre, during and post by stack and event → download plan

In [ ]:
def txt(v):
    return "FALTA" if v is None or (not isinstance(v, str) and pd.isna(v)) or v == "" else str(v)

sel = pasadas[pasadas["orbita"].isin(ORBITAS_A_BAJAR)]
if STACKS_A_BAJAR:
    sel = sel[sel["stack"].isin(STACKS_A_BAJAR)]

filas = []
for (stack, orbita), p in sel.groupby(["stack", "orbita"]):
    libres = p[~p["en_ventana"] & ~p["justo_antes"]].sort_values("dia")
    for i, ((ini, fin), lugar) in enumerate(zip(VENTANAS, EVENTOS), start=1):
        dentro = p[(p["dia"] >= ini) & (p["dia"] <= fin + timedelta(days=1))]
        previas = libres[libres["dia"] < ini - MARGEN_PREVIO]
        posteriores = libres[(libres["dia"] > fin + timedelta(days=1)) &
                             (libres["dia"] <= fin + timedelta(days=POST_MAX_DIAS))]
        if PREFERIR_DOBLE_POL:
            previas = previas[~previas["pols"].str.contains("SHSH")] if (~previas["pols"].str.contains("SHSH")).any() else previas
            posteriores = posteriores[~posteriores["pols"].str.contains("SHSH")] if (~posteriores["pols"].str.contains("SHSH")).any() else posteriores
        pre = previas.iloc[-1] if len(previas) else None
        post = posteriores.iloc[0] if len(posteriores) else None
        filas.append({
            "stack": stack, "orbita": orbita, "evento": f"E{i}", "fecha_evento": ini.date(), "lugar": lugar,
            "pre": pre["dia"].date() if pre is not None else None,
            "durante": ", ".join(f"{d:%Y-%m-%d}" for d in dentro["dia"]) or None,
            "post": post["dia"].date() if post is not None else None,
            "archivos_pre": ";".join(pre["archivos"]) if pre is not None else "",
            "archivos_durante": ";".join(a for lst in dentro["archivos"] for a in lst),
            "archivos_post": ";".join(post["archivos"]) if post is not None else "",
        })

fases = pd.DataFrame(filas)
fases["completo"] = fases[["pre", "durante", "post"]].notna().all(axis=1)
fases.to_csv(DIR_META / "fases_eventos.csv", index=False)

for (stack, orbita), grupo in fases.groupby(["stack", "orbita"]):
    print(SEP)
    print(f"Stack {stack}  ·  órbita {orbita.upper()}")
    print(f"{'Evento':<7}{'Fecha':<12}{'Pre':<12}{'Durante':<24}{'Post':<12}")
    for r in grupo.itertuples():
        print(f"{r.evento:<7}{str(r.fecha_evento):<12}{txt(r.pre):<12}{txt(r.durante):<24}{txt(r.post):<12}")
    print(f"{'Completos':<10}: {grupo['completo'].sum()} de {len(grupo)}")

usar = fases[fases["durante"].notna()]
nombres = set()
for col in ("archivos_pre", "archivos_durante", "archivos_post"):
    for v in usar[col]:
        nombres |= set(x for x in v.split(";") if x)
plan = inv[inv["archivo"].isin(nombres)].sort_values(["stack", "fecha", "frame"]).reset_index(drop=True)
plan.to_csv(DIR_META / "plan_descarga.csv", index=False)

print(SEP)
print(f"{'Archivos en el plan':<22}: {len(plan)}")
for (st_, orb), n in plan.groupby(["stack", "orbita"]).size().items():
    print(f"{'  ' + st_ + ' (' + orb + ')':<22}: {n}")
print(f"{'Tamaño del plan':<22}: {plan['gb'].sum(min_count=1):.1f} GB" if plan["gb"].notna().any() else f"{'Tamaño del plan':<22}: (ASF no lo reporta)")
print(SEP)

Stack A_T047  ·  órbita ASCENDENTE
Evento Fecha       Pre         Durante                 Post        
E1     2026-01-19  FALTA       FALTA                   FALTA       
E2     2026-02-01  FALTA       FALTA                   FALTA       
E3     2026-05-04  FALTA       FALTA                   FALTA       
E4     2026-05-29  FALTA       FALTA                   2026-06-17  
E5     2026-07-19  2026-07-11  2026-07-23              2026-08-16  
E6     2026-08-28  2026-08-16  2026-08-28, 2026-09-09  2026-09-21  
Completos : 2 de 6
Stack A_T148  ·  órbita ASCENDENTE
Evento Fecha       Pre         Durante                 Post        
E1     2026-01-19  FALTA       FALTA                   FALTA       
E2     2026-02-01  FALTA       FALTA                   FALTA       
E3     2026-05-04  FALTA       FALTA                   FALTA       
E4     2026-05-29  FALTA       FALTA                   2026-06-24  
E5     2026-07-19  2026-07-06  FALTA                   2026-08-11  
E6     2026-08-28  2026-08-

## 6. Check what is already downloaded

In [ ]:
import os
import re
import time
import pandas as pd
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import h5py

GRUPO = "/science/LSAR/GCOV/grids/frequencyA"
BLOQUEO = re.compile(r"unable to lock|file locking|permission denied|acceso denegado|being used by another|siendo utilizado", re.I)

if "plan" not in globals():
    plan = pd.read_csv(DIR_META / "plan_descarga.csv", parse_dates=["fecha"])
    print(f"Se cargó plan_descarga.csv ({len(plan)} archivos); no hizo falta iniciar sesión.")


def abrir_h5(ruta):
    try:
        return h5py.File(ruta, "r", locking=False)
    except (TypeError, ValueError):
        return h5py.File(ruta, "r")


def estado_archivo(ruta, gb_esperado):
    if not ruta.exists():
        return "falta"
    if pd.notna(gb_esperado) and ruta.stat().st_size / 1e9 < gb_esperado - 0.001:
        return "incompleto (tamaño)"
    try:
        with abrir_h5(ruta) as f:
            return "ok" if GRUPO in f else "incompleto (sin datos)"
    except PermissionError:
        return "en uso"
    except OSError as e:
        if BLOQUEO.search(str(e)):
            return "en uso"
        return "incompleto (no abre)"


def apartar(ruta):
    for intento in range(3):
        try:
            if BORRAR_INCOMPLETOS:
                ruta.unlink()
            else:
                DIR_MALOS.mkdir(exist_ok=True)
                ruta.replace(DIR_MALOS / ruta.name)
            return True
        except PermissionError:
            time.sleep(2)
    return False


existentes = {r.archivo: (DIR_H5 / r.archivo).stat().st_size
              for r in plan.itertuples() if (DIR_H5 / r.archivo).exists()}
if existentes:
    time.sleep(3)
creciendo = {a for a, t in existentes.items()
             if (DIR_H5 / a).exists() and (DIR_H5 / a).stat().st_size != t}

plan["estado"] = ["descargándose" if r.archivo in creciendo else estado_archivo(DIR_H5 / r.archivo, r.gb)
                  for r in plan.itertuples()]

for i, r in plan[plan["estado"].str.startswith("incompleto")].iterrows():
    plan.at[i, "estado"] = "apartado" if apartar(DIR_H5 / r["archivo"]) else "en uso"

print(SEP)
for est, n in plan["estado"].value_counts().items():
    print(f"{est:<26}: {n}")
gb_ok = plan.loc[plan["estado"] == "ok", "gb"].sum()
gb_rest = plan.loc[plan["estado"] != "ok", "gb"].sum()
print(f"{'Ya completos':<26}: {gb_ok:.1f} GB")
print(f"{'Por bajar':<26}: {gb_rest:.1f} GB")
print(SEP)
if (plan["estado"] == "apartado").any():
    accion = "borrados" if BORRAR_INCOMPLETOS else f"movidos a {DIR_MALOS.name}/"
    print(f"Incompletos {accion}; se vuelven a bajar en la sección 7.")
    if not BORRAR_INCOMPLETOS:
        print(f"Cuando terminen, pueden borrar {DIR_MALOS} para liberar espacio.")
bloqueados = plan[plan["estado"].isin(["en uso", "descargándose"])]
if len(bloqueados):
    print(f"{len(bloqueados)} archivo(s) que no se tocaron porque otro programa los tiene abiertos o los está escribiendo:")
    for r in bloqueados.itertuples():
        print(f"  {r.estado:<14} {r.archivo[:85]}")
    print("Cierren los otros notebooks o los python.exe que queden de descargas anteriores, y corran de nuevo esta celda.")
partes = sorted(DIR_H5.glob("*.part"))
if partes:
    print(f"Hay {len(partes)} descarga(s) a medias (.part); la sección 7 las reemplaza.")

ok                        : 34
Ya completos              : 177.7 GB
Por bajar                 : 0.0 GB


## 7. Download what is missing

In [ ]:
faltan = plan[plan["estado"].isin(["falta", "apartado"])].reset_index(drop=True)
gb_faltan = faltan["gb"].sum()
print(f"{'Archivos por bajar':<20}: {len(faltan)}")
print(f"{'Tamaño':<20}: {gb_faltan:.1f} GB" if faltan["gb"].notna().all() else f"{'Tamaño':<20}: al menos {gb_faltan:.1f} GB (ASF no reporta todos)")

if gb_faltan > MAX_DESCARGA_GB:
    raise RuntimeError(f"Faltan {gb_faltan:.1f} GB > MAX_DESCARGA_GB={MAX_DESCARGA_GB}. "
                       "Suban el límite o limiten ORBITAS_A_BAJAR / STACKS_A_BAJAR.")
if len(faltan) and "session" not in globals():
    raise RuntimeError("Falta iniciar sesión: corran la sección 2 antes de descargar.")


def bajar(fila):
    final = DIR_H5 / fila.archivo
    parcial = DIR_H5 / (fila.archivo + ".part")
    if parcial.exists():
        try:
            parcial.unlink()
        except PermissionError:
            return "en uso (el .part lo tiene abierto otro programa)"
    t0 = time.time()
    asf.download_url(url=fila.url, path=str(DIR_H5), filename=parcial.name, session=session)
    if not parcial.exists():
        return "error (no se creó el archivo)"
    est = estado_archivo(parcial, fila.gb)
    if est != "ok":
        return f"error al verificar ({est}); se dejó como .part"
    gb = parcial.stat().st_size / 1e9
    parcial.replace(final)
    return f"ok ({gb:.2f} GB en {(time.time() - t0) / 60:.1f} min)"


resultados = []
for k, fila in enumerate(faltan.itertuples(), start=1):
    print(f"[{k}/{len(faltan)}] {fila.archivo[:72]}  ({fila.gb:.2f} GB)", flush=True)
    try:
        r = bajar(fila)
    except KeyboardInterrupt:
        print("      -> interrumpido. El archivo a medias quedó como .part; vuelvan a correr esta celda para seguir.")
        raise
    except Exception as e:
        r = f"error ({type(e).__name__}: {str(e)[:150]})"
        if "401" in str(e) or "Unauthorized" in str(e) or "approve_app" in str(e):
            r += "  -> revisen la sesión (sección 2) y la autorización de ASF"
    print(f"      -> {r}", flush=True)
    resultados.append(r)

plan["estado"] = [estado_archivo(DIR_H5 / r.archivo, r.gb) for r in plan.itertuples()]
print(SEP)
for est, n in plan["estado"].value_counts().items():
    print(f"{est:<26}: {n}")
print(SEP)
if (plan["estado"] != "ok").any():
    print("Quedaron archivos sin terminar: corran la sección 6 y luego esta otra vez.")
else:
    print("Todos los archivos del plan están completos.")

Archivos por bajar  : 0
Tamaño              : 0.0 GB
ok                        : 34
Todos los archivos del plan están completos.


## 8. Confirm the orbit with the internal metadata

In [10]:
IDENT = "/science/LSAR/identification"

def leer_texto(grupo, clave):
    if clave not in grupo:
        return ""
    v = grupo[clave][()]
    v = v.item() if hasattr(v, "item") and getattr(v, "size", 1) == 1 else v
    return (v.decode() if isinstance(v, bytes) else str(v)).strip()

meta = []
for r in plan[plan["estado"] == "ok"].itertuples():
    d = {"archivo": r.archivo, "orbita_h5": "", "mira": "", "track_h5": "", "frame_h5": "", "pols_h5": ""}
    with abrir_h5(DIR_H5 / r.archivo) as f:
        g = f[GRUPO]
        d["pols_h5"] = ",".join(k for k in g.keys() if len(k) == 4 and k.isupper() and k[:2] == k[2:])
        if IDENT in f:
            idt = f[IDENT]
            d["orbita_h5"] = leer_texto(idt, "orbitPassDirection")
            d["mira"]      = leer_texto(idt, "lookDirection")
            d["track_h5"]  = leer_texto(idt, "trackNumber")
            d["frame_h5"]  = leer_texto(idt, "frameNumber")
    meta.append(d)

if not meta:
    print("No hay archivos completos todavía.")
else:
    locales = plan.merge(pd.DataFrame(meta), on="archivo")
    def chequeo(r):
        if not r["orbita_h5"]:
            return "sin metadato (se usa el nombre)"
        return "OK" if r["orbita_h5"].lower().startswith("asc") == (r["dir"] == "A") else "NO COINCIDE"
    locales["chequeo_orbita"] = locales.apply(chequeo, axis=1)

    print(SEP)
    print(f"{'Fecha':<12}{'Stack':<8}{'Frame':<7}{'Órbita':<13}{'En el .h5':<12}{'Mira':<7}{'Pols':<11}Chequeo")
    for r in locales.itertuples():
        print(f"{r.fecha:%Y-%m-%d}  {r.stack:<8}F{r.frame:03d}   {r.orbita:<13}{(r.orbita_h5 or '-'):<12}"
              f"{(r.mira or '-'):<7}{r.pols_h5:<11}{r.chequeo_orbita}")
    print(SEP)
    locales.to_csv(DIR_META / "archivos_descargados.csv", index=False)
    print(f"{'Guardado':<10}: archivos_descargados.csv")

Fecha       Stack   Frame  Órbita       En el .h5   Mira   Pols       Chequeo
2026-07-11  A_T047  F004   Ascendente   Ascending   Left   HHHH,HVHV  OK
2026-07-11  A_T047  F005   Ascendente   Ascending   Left   HHHH,HVHV  OK
2026-07-23  A_T047  F004   Ascendente   Ascending   Left   HHHH,HVHV  OK
2026-07-23  A_T047  F005   Ascendente   Ascending   Left   HHHH,HVHV  OK
2026-08-16  A_T047  F005   Ascendente   Ascending   Left   HHHH,HVHV  OK
2026-08-28  A_T047  F004   Ascendente   Ascending   Left   HHHH,HVHV  OK
2026-08-28  A_T047  F005   Ascendente   Ascending   Left   HHHH,HVHV  OK
2026-09-09  A_T047  F004   Ascendente   Ascending   Left   HHHH,HVHV  OK
2026-09-09  A_T047  F005   Ascendente   Ascending   Left   HHHH,HVHV  OK
2026-09-21  A_T047  F004   Ascendente   Ascending   Left   HHHH,HVHV  OK
2026-09-21  A_T047  F005   Ascendente   Ascending   Left   HHHH,HVHV  OK
2026-08-11  A_T148  F004   Ascendente   Ascending   Left   HHHH,HVHV  OK
2026-08-11  A_T148  F005   Ascendente   Ascend